# ITMA Head Pretraining (3-stage memory-aware curriculum)

Runs `scripts/pretrain_itma.py` on Colab T4. Total time: ~10-15 min for ~2k triples.

Output: `checkpoints/itma_head_v4.pt` — drop into the repo (overwrites `itma_head.pt` if you confirm via eval).

**Prereqs:** GitHub access (repo URL below) and a `GROQ_API_KEY` if you want to regenerate triples for more sources. If you skip regeneration the run will still work but Stage C (unhelpful memory) will be a no-op when the data has only 1 source.

## 0. Runtime check (T4 GPU)

In [ ]:
!nvidia-smi

## 1. Get the code

Two options.

**A. Mount Google Drive** (simplest — keeps your local checkpoints in sync). Put a copy of the `lecture_rag_agent` folder under `MyDrive/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
REPO = '/content/drive/MyDrive/lecture_rag_agent'
assert os.path.isdir(REPO), f'Not found: {REPO}. Upload repo to Drive or use option B.'
%cd $REPO
!pwd && ls

**B. Clone from GitHub** (replace the URL with your fork).

In [ ]:
# !git clone https://github.com/<your-user>/lecture_rag_agent.git /content/lecture_rag_agent
# %cd /content/lecture_rag_agent
# REPO = '/content/lecture_rag_agent'

## 2. Install dependencies

In [ ]:
!pip install -q sentence-transformers torch numpy rank_bm25 python-dotenv requests huggingface_hub

## 3. (Optional) Regenerate triples across all 5 transcripts

Skip this cell if `data/itma_pretrain/triples.jsonl` already covers >=2 sources.
Set `GROQ_API_KEY` (and optionally `HF_API_KEY`) in a `.env` at the repo root before running.

Wall-clock: ~20-30 min for 5 transcripts at default rate-limit delays.

In [ ]:
# from getpass import getpass
# import os
# os.environ['GROQ_API_KEY'] = getpass('GROQ_API_KEY: ')
# !python scripts/build_itma_pretrain_data.py --max-chunks 60

## 4. Inspect triples (sanity check sources)

In [ ]:
import json
from collections import Counter
src_counts = Counter()
with open('data/itma_pretrain/triples.jsonl', encoding='utf-8') as f:
    for line in f:
        t = json.loads(line)
        src_counts[t.get('source', '_unknown')] += 1
print(f'total triples: {sum(src_counts.values())}')
print(f'sources: {dict(src_counts)}')

## 5. Run pretraining (3-stage curriculum)

Defaults: 5 epochs per stage, batch 64, Adam. The script auto-detects CUDA.

In [ ]:
!python scripts/pretrain_itma.py \
    --triples data/itma_pretrain/triples.jsonl \
    --out checkpoints/itma_head_v4.pt \
    --epochs-a 5 --epochs-b 5 --epochs-c 5 \
    --batch-size 64 --seed 42

## 6. Inspect training log

In [ ]:
import json
log = json.load(open('checkpoints/itma_head_v4.train_log.json'))
import pandas as pd
df = pd.DataFrame(log)
print(df[['stage','epoch','total_loss','margin_loss','gate_mean','score_pos_mean','score_neg_mean']].to_string(index=False))

In [ ]:
# Plot gate evolution across stages
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for stage in ['A','B','C']:
    sub = df[df.stage == stage]
    if len(sub) == 0:
        continue
    ax[0].plot(sub.epoch, sub.gate_mean, marker='o', label=f'Stage {stage}')
    ax[1].plot(sub.epoch, sub.total_loss, marker='o', label=f'Stage {stage}')
ax[0].set_ylabel('mean sigmoid(gate)'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].set_title('Gate value')
ax[1].set_ylabel('total loss'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].set_title('Loss')
plt.tight_layout(); plt.show()

## 7. Download the checkpoint

If you mounted Drive in step 1A the checkpoint is already saved to `MyDrive/lecture_rag_agent/checkpoints/`. Skip the cell below.

Otherwise (option B / clone), run this to pull the file to your local machine:

In [ ]:
# from google.colab import files
# files.download('checkpoints/itma_head_v4.pt')
# files.download('checkpoints/itma_head_v4.train_log.json')

## 8. Next step (run locally)

Drop `itma_head_v4.pt` into `checkpoints/` on your local repo, then run the cold-start eval **with ID-boost disabled** to see whether the gate is now doing real work:

```bash
python scripts/cold_start_eval.py \
    --checkpoint-itma checkpoints/itma_head_v4.pt \
    --systems itma_no_boost \
    --out analysis/cold_start_v4_no_boost.csv
```

Compare against the old `itma_no_boost` (flat 0.8305). If the new run climbs with N, the gate is finally adapting and the ID-boost hack can be removed from `src/itma/integration.py`.